# Prompt Templates & Output Parsers
### Build reusable, dynamic prompts and parse structured outputs

---
**Topics Covered:**
- `PromptTemplate` for single-variable and multi-variable prompts
- `ChatPromptTemplate` with system + user roles
- `FewShotChatMessagePromptTemplate` for in-context examples
- `StrOutputParser` — strip `AIMessage` wrapper to plain text
- `JsonOutputParser` — force structured JSON responses
- Composing prompts into a simple LCEL pipeline with `|`


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq", temperature=0.6)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


ImportError: Initializing ChatGroq requires the langchain-groq package. Please install it with `pip install langchain-groq`

## 1. `PromptTemplate` — Simple String Templates

Use curly-brace placeholders `{variable}` to inject values at call time.

In [ ]:
from langchain_core.prompts import PromptTemplate

explain_template = PromptTemplate(
    input_variables=["concept", "audience"],
    template="Explain {concept} to a {audience} in no more than 100 words.",
)

# Render the template — no LLM call yet
rendered = explain_template.format(concept="neural networks", audience="10-year-old")
print(rendered)

In [ ]:
# Now invoke the LLM with the rendered prompt
response = llm.invoke(rendered)
print(response.content)

## 2. `ChatPromptTemplate` — Role-Aware Templates

Use `("system", ...)` and `("human", ...)` tuples to assign message roles declaratively.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

blog_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical blog writer who writes engaging, SEO-friendly posts."),
    ("human", "Write a 150-word intro paragraph for a blog post titled: '{title}'.")
])

# Format and inspect before calling the model
messages = blog_prompt.format_messages(title="Why Python Type Hints Will Save Your Sanity")
for m in messages:
    print(f"[{m.type.upper()}]:", m.content[:80], "...")

In [ ]:
result = llm.invoke(messages)
print(result.content)

## 3. `StrOutputParser` — Clean String Output

Chain a parser with `|` to strip the `AIMessage` envelope and return a plain `str`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Build a minimal pipeline: template → llm → parser
haiku_prompt = ChatPromptTemplate.from_messages([
    ("system", "You only reply with haiku poetry. Strictly 5-7-5 syllables."),
    ("human", "Write a haiku about {topic}.")
])

chain = haiku_prompt | llm | StrOutputParser()

# chain.invoke returns a plain string now
poem = chain.invoke({"topic": "machine learning overfitting"})
print(type(poem))   # <class 'str'>
print(poem)

## 4. `JsonOutputParser` — Structured Data Extraction

Ask the model to return JSON and automatically parse it into a Python dict.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You respond ONLY with valid JSON. No markdown fences, no extra text.\n"
        "Schema: {{\"name\": str, \"founded\": int, \"hq\": str, \"employees_k\": float}}"
    )),
    ("human", "Give me the company profile for {company}.")
])

json_chain = json_prompt | llm | JsonOutputParser()

profile = json_chain.invoke({"company": "DeepMind"})
print(type(profile))   # <class 'dict'>
print(profile)

In [ ]:
# Access fields directly like a normal Python dict
print("Company  :", profile["name"])
print("Founded  :", profile["founded"])
print("HQ       :", profile["hq"])

## 5. Few-Shot Prompting with `ChatPromptTemplate`

Provide worked examples inside the prompt to steer the model toward a specific output format.

In [ ]:
# Manual few-shot via list of messages
few_shot_messages = [
    ("system", "You classify the sentiment of a code review comment as POSITIVE, NEGATIVE, or NEUTRAL."),
    ("human",  "Comment: 'This function is clean and well-named.'"),
    ("ai",     "POSITIVE"),
    ("human",  "Comment: 'Magic numbers everywhere — please use named constants.'"),
    ("ai",     "NEGATIVE"),
    ("human",  "Comment: 'Added unit test for the edge case.'"),
    ("ai",     "POSITIVE"),
    ("human",  "Comment: '{comment}'"),
]

classifier_prompt = ChatPromptTemplate.from_messages(few_shot_messages)
classifier_chain  = classifier_prompt | llm | StrOutputParser()

samples = [
    "Indentation is inconsistent throughout the file.",
    "Good use of list comprehension here.",
    "I've reviewed this PR.",
]

for sample in samples:
    label = classifier_chain.invoke({"comment": sample})
    print(f"{label:10s} → {sample}")

## 6. Multi-Variable Chat Template — Code Review Bot

In [ ]:
review_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a senior {language} engineer. Review the code snippet below.\n"
        "Focus on: correctness, readability, and performance. Keep your review under 120 words."
    )),
    ("human", "```{language}\n{code}\n```")
])

review_chain = review_prompt | llm | StrOutputParser()

code_snippet = """
def find_duplicates(lst):
    duplicates = []
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):
            if lst[i] == lst[j] and lst[i] not in duplicates:
                duplicates.append(lst[i])
    return duplicates
"""

feedback = review_chain.invoke({"language": "Python", "code": code_snippet})
print(feedback)

---
### Summary

| Class | Purpose |
|-------|---------|
| `PromptTemplate` | Single-string template with `{vars}` |
| `ChatPromptTemplate` | Role-structured template `("system",...), ("human",...)`|
| `StrOutputParser` | `AIMessage` → `str` |
| `JsonOutputParser` | `AIMessage` → `dict` |
| `chain = prompt \| llm \| parser` | LCEL pipeline |

**Next → `03_tool_calling.ipynb`**